# Generation of a csv file for downloading from PubChem

### Content   <a name="content"></a>

1. [Load and analyse PubChem BioAssay](#1)
2. [Create a data frame with CIDs, SIDs, SMILES and targets](#2)
3. [Balance dataset taking any 4th sample from target 1](#3)
4. [Create a csv file for extraction of atom coordinate from PubChem](#4)

### Load and analyse PubChem BioAssay data <a name="1"></a>

In [2]:
import pandas as pd 

# load the PubChem AID 449704 bioassy dataset
# https://pubchem.ncbi.nlm.nih.gov/bioassay/449704
df = pd.read_csv('AID_449704_datatable.csv', low_memory=False, on_bad_lines='skip')

# To avoid truncation of some columns during data frame display
pd.set_option('display.max_columns', None) 

# Display the data frame
print('Shape of the data frame: ', df.shape)
df.head()

Shape of the data frame:  (5697, 14)


,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,PubChem Standard Value,Standard Type,Standard Relation,Standard Value,Standard Units,Activity Comment
0,1,103163899,2807642.0,CCC1=CC2=C(S1)N=CN=C2SC3=NN=C(S3)N,Active,NaN,NaN,NaN,0.54000,EC50,=,540.00,nM,NaN
1,2,103164645,3239884.0,CCOC(=O)C1=CC(=CC=C1)N2C(=NC(=NC2(C)C)N)N,Unspecified,NaN,NaN,NaN,1.25000,EC50,>,1250.00,nM,NaN
2,3,103164647,98652.0,C1=CC=C2C=C(C=CC2=C1)S(=O)(=O)C3=CC4=C(C=C3)N=...,Active,NaN,NaN,NaN,1.70900,EC50,=,1709.00,nM,NaN
3,4,103164913,198062.0,CC1=C2C(=CC=C1)N=C(N=C2N)N,Unspecified,NaN,NaN,NaN,1.25000,EC50,>,1250.00,nM,NaN
4,5,103165594,4993.0,CCC1=C(C(=NC(=N1)N)N)C2=CC=C(C=C2)Cl,Active,NaN,NaN,NaN,0.01456,EC50,=,14.56,nM,NaN


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5697 entries, 0 to 5696
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   PUBCHEM_RESULT_TAG             5697 non-null   int64  
 1   PUBCHEM_SID                    5697 non-null   int64  
 2   PUBCHEM_CID                    5694 non-null   float64
 3   PUBCHEM_EXT_DATASOURCE_SMILES  5694 non-null   object 
 4   PUBCHEM_ACTIVITY_OUTCOME       5697 non-null   object 
 5   PUBCHEM_ACTIVITY_SCORE         0 non-null      float64
 6   PUBCHEM_ACTIVITY_URL           0 non-null      float64
 7   PUBCHEM_ASSAYDATA_COMMENT      13 non-null     object 
 8   PubChem Standard Value         5684 non-null   float64
 9   Standard Type                  5697 non-null   object 
 10  Standard Relation              5684 non-null   object 
 11  Standard Value                 5684 non-null   float64
 12  Standard Units                 5684 non-null   o

In [4]:
# Remive missing values
df = df[df['PUBCHEM_CID'].notna()]
df.shape

(5694, 14)

In [5]:
# Remove duplicates without keeping a sample ofthem and reset the indexes 
df = df.drop_duplicates(subset='PUBCHEM_CID', keep=False).reset_index(drop=True)
df.shape

(5524, 14)

In [6]:
# Turn CID float data type into integer
df['PUBCHEM_CID'] = df['PUBCHEM_CID'].astype('int64') 

In [7]:
# Filter inhibitors 
df = df[['PUBCHEM_CID',
         'PUBCHEM_SID',
         'PUBCHEM_EXT_DATASOURCE_SMILES',
         'PUBCHEM_ACTIVITY_OUTCOME']] 
df.shape # 646675

(5524, 4)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5524 entries, 0 to 5523
Data columns (total 4 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   PUBCHEM_CID                    5524 non-null   int64 
 1   PUBCHEM_SID                    5524 non-null   int64 
 2   PUBCHEM_EXT_DATASOURCE_SMILES  5524 non-null   object
 3   PUBCHEM_ACTIVITY_OUTCOME       5524 non-null   object
dtypes: int64(2), object(2)
memory usage: 172.8+ KB


[<a href="#content">Back to top</a>]

## Create a data frame with CIDs, SIDs, SMILES and targets <a name="2"></a>

In [9]:
df['PUBCHEM_ACTIVITY_OUTCOME'] = df['PUBCHEM_ACTIVITY_OUTCOME'].astype(str) 
# Get unique values from 'column1'
unique_values = df['PUBCHEM_ACTIVITY_OUTCOME'].unique()
unique_values 

array(['Active', 'Unspecified', 'Inconclusive'], dtype=object)

In [10]:
df.rename(columns={'PUBCHEM_CID':'CID',
                   'PUBCHEM_SID':'SID',
                   'PUBCHEM_EXT_DATASOURCE_SMILES' : 'SMILES',
                   'PUBCHEM_ACTIVITY_OUTCOME':'target'}, inplace=True)

# Set the option to explicitly handle downcasting
pd.set_option('future.no_silent_downcasting', True)

# Create a mapping dictionary to replace string values with numeric values
mapping = {'Active': 1, 'Unspecified':0, 'Inconclusive':2}

# Replace string values with numeric values using the mapping dictionary
df['target'] = df['target'].replace(mapping)

# Display the data frame
print('Shape of the data frame: ', df.shape)
df.head()

Shape of the data frame:  (5524, 4)


,CID,SID,SMILES,target
0,2807642,103163899,CCC1=CC2=C(S1)N=CN=C2SC3=NN=C(S3)N,1
1,3239884,103164645,CCOC(=O)C1=CC(=CC=C1)N2C(=NC(=NC2(C)C)N)N,0
2,98652,103164647,C1=CC=C2C=C(C=CC2=C1)S(=O)(=O)C3=CC4=C(C=C3)N=...,1
3,198062,103164913,CC1=C2C(=CC=C1)N=C(N=C2N)N,0
4,4993,103165594,CCC1=C(C(=NC(=N1)N)N)C2=CC=C(C=C2)Cl,1


In [11]:
# Turn CID float data type into integer
df['target'] = pd.to_numeric(df['target']) 

In [12]:
df['target'].value_counts()

target
1    4529
0     993
2       2
Name: count, dtype: int64

In [13]:
df.to_csv('HF_malaria_AID499704_CIDs_SIDs_SMILESs_targets.csv') 

In [14]:
# Remove target = 2 
df = df[df['target'] != 2]

[<a href="#content">Back to top</a>]

## Balance dataset taking any 4th sample from target 1 <a name="3"></a>

In [15]:
# Handle the part of target 0 in the test set 
df_0 = df[df['target']==0]

In [16]:
# Handle the part of target 1 in the test set 
df_1 = df[df['target']==1]

# Shuffle the resulting data set
df_1 = df_1.sample(
    frac = 1,        # Return entire dataframe
    random_state=1   # Make result reproducible
    ).reset_index(drop=True)

# Extract every 4th row of samples labled 0   
n = 4
df_1 = df_1[df_1.index % n == 1] 
df_1.shape

(1132, 4)

In [17]:
# Combine target 1 and 0
df = pd.concat([df_0, df_1], axis=0)

# Shuffle the resulting data set
df = df.sample(
    frac = 1,        # Return entire dataframe
    random_state=1   # Make result reproducible
    ).reset_index(drop=True)

print(df.shape)
df.head()

(2125, 4)


,CID,SID,SMILES,target
0,4169706,103712074,C1C(OC2=C(C1=O)C=C(C=C2)F)C3=CC=CC=C3,0
1,13076,103713284,CC1=C2C=CC=CC2=C(C3=CC=CC=C13)C,1
2,23885521,103713805,CC1=CC=CC=C1N(CC(=O)NCC2CCCO2)C(=O)C3=C(C(=NS3...,0
3,588066,103711416,CN1C2=C(C(=O)N(C1=O)C)N(C=N2)CC(CNC3=NCCCC3)O,0
4,16018404,103712702,COC1=CC=CC(=C1)C2=NOC(=C2)C(=O)NC3=C(C=CC(=C3)...,1


[<a href="#content">Back to top</a>]

## Create a csv file for extraction of atom coordinate from PubChem  <a name="4"></a>

In [18]:
df.to_csv('AID499704_CIDs_SIDs_SMILESs_targets.csv', index=False) 